# Stock ML Dynamic — Iterative feature and target experiments

Train models here, not in Streamlit. Run from top to bottom. Saved experiment artifacts are displayed by the dashboard. This notebook does not modify PostgreSQL.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from stock_ml_dynamic.db import engine, price_history
from stock_ml_dynamic.experiment import ExperimentSpec, AVAILABLE_FEATURES
from stock_ml_dynamic.experiments import build_experiment_frame
from stock_ml_dynamic.experiment_training import train_experiment, save_experiment, list_experiments, comparison_rows


In [ ]:
import sys
import pandas as pd
import sklearn
import ipykernel

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("IPyKernel:", ipykernel.__version__)

%load_ext autoreload
%autoreload 2

Python: /home/oem/Documents/github/stock_ml_dynamic/.venv/bin/python
Pandas: 2.3.3
Scikit-learn: 1.9.1
IPyKernel: 7.4.0


## 1. Select stock and historical window
Change these values and rerun the cells below.

In [2]:
SYMBOL = "AAPL"
INTERVAL = "1d"
BARS = 2000
with engine().connect() as conn:
    prices = price_history(conn, SYMBOL, INTERVAL, BARS)
print(f"Loaded {len(prices)} rows for {SYMBOL} / {INTERVAL}")
display(prices.tail())


Loaded 2000 rows for AAPL / 1d


,time,open,high,low,close,adj_close,volume
1995,2026-11-02 00:00:00+00:00,342.670013,343.799988,335.820007,338.320007,338.320007,64757700
1996,2026-11-03 00:00:00+00:00,325.579987,334.500000,325.329987,331.899994,331.899994,63160000
1997,2026-11-04 00:00:00+00:00,341.570007,354.500000,340.989990,349.510010,349.510010,89637700
1998,2026-11-05 00:00:00+00:00,353.890015,355.380005,342.059998,344.170013,344.170013,75678900
1999,2026-11-06 00:00:00+00:00,338.730011,345.790009,338.299988,342.369995,342.369995,44676300


## 2. Define feature selection and calculation parameters
The feature names remain stable even if indicator windows change; each experiment saves the exact parameters. Edit this cell to change the feature set.

In [3]:
FEATURES = list(AVAILABLE_FEATURES)
# Example ablation: FEATURES = [f for f in AVAILABLE_FEATURES if f not in ("macd_ratio", "rsi_14")]
SMA_SHORT = 10
SMA_MEDIUM = 20
SMA_LONG = 50
RSI_WINDOW = 14
VOLATILITY_WINDOW = 20
VOLUME_WINDOW = 20


## 3. Define the target
`direction`: 1 when future close > current close. `threshold`: 1 when forward return > THRESHOLD (fractional, 0.01 = 1%). `return`: continuous forward return (regression). HORIZON is measured in bars, not calendar days.

In [4]:
TARGET_TYPE = "direction"  # "direction", "threshold", or "return"
HORIZON = 5
THRESHOLD = 0.01
TEST_FRACTION = 0.20
spec = ExperimentSpec(features=tuple(FEATURES), horizon=HORIZON, target_type=TARGET_TYPE, threshold=THRESHOLD, test_fraction=TEST_FRACTION, sma_short=SMA_SHORT, sma_medium=SMA_MEDIUM, sma_long=SMA_LONG, rsi_window=RSI_WINDOW, volatility_window=VOLATILITY_WINDOW, volume_window=VOLUME_WINDOW).validate()
print(spec.as_dict())


{'features': ['return_1', 'return_5', 'return_20', 'sma_10_ratio', 'sma_20_ratio', 'sma_50_ratio', 'volatility_20', 'volume_ratio_20', 'rsi_14', 'macd_ratio', 'range_ratio', 'body_ratio'], 'horizon': 5, 'target_type': 'direction', 'threshold': 0.01, 'test_fraction': 0.2, 'min_labeled_rows': 200, 'random_state': 42, 'sma_short': 10, 'sma_medium': 20, 'sma_long': 50, 'volatility_window': 20, 'volume_window': 20, 'rsi_window': 14}


## 4. Inspect features and targets before training
All features are computed using current/past candles; targets use future closes. Last HORIZON rows have no label.

In [5]:
frame = build_experiment_frame(prices, spec)
display(frame[["time", *FEATURES, "target_forward_return", "target"]].tail(12))
print("Labeled rows:", frame.target.notna().sum())
print("Target distribution:")
print(frame.target.dropna().describe() if TARGET_TYPE == "return" else frame.target.value_counts(dropna=False))


,time,return_1,return_5,return_20,sma_10_ratio,sma_20_ratio,sma_50_ratio,volatility_20,volume_ratio_20,rsi_14,macd_ratio,range_ratio,body_ratio,target_forward_return,target
1988,2026-10-22 00:00:00+00:00,0.012297,0.018187,0.036616,0.017607,0.028493,0.059652,0.010791,1.233073,64.751053,0.012038,0.017460,0.009509,0.001982,1.0
1989,2026-10-23 00:00:00+00:00,-0.001407,0.012432,0.019527,0.012892,0.026037,0.055433,0.010330,1.026712,63.918487,0.012568,0.017197,-0.003981,-0.008915,0.0
1990,2026-10-26 00:00:00+00:00,-0.010986,0.002828,0.016283,0.000908,0.013941,0.041437,0.010495,1.095789,57.690631,0.012084,0.017621,-0.000029,-0.016254,0.0
1991,2026-10-27 00:00:00+00:00,0.009276,0.022386,0.053734,0.008481,0.020683,0.048461,0.008415,1.023078,61.099272,0.012085,0.015529,0.011452,-0.043791,0.0
1992,2026-10-28 00:00:00+00:00,-0.009421,-0.000465,0.032461,-0.001272,0.009462,0.036467,0.008586,0.930817,56.109031,0.011391,0.010645,-0.008907,0.016520,1.0
1993,2026-10-29 00:00:00+00:00,0.014775,0.001982,0.056279,0.011454,0.021589,0.049751,0.008743,1.100809,61.385415,0.011634,0.011837,0.002615,-0.013585,0.0
1994,2026-10-30 00:00:00+00:00,-0.012267,-0.008915,0.032785,-0.001292,0.007444,0.034798,0.009172,1.678803,55.348310,0.010978,0.043061,-0.006028,-0.006558,0.0
1995,2026-11-02 00:00:00+00:00,-0.018309,-0.016254,0.016312,-0.018264,-0.011786,0.014087,0.010177,1.544478,47.873070,0.008928,0.023587,-0.012694,NaN,NaN
1996,2026-11-03 00:00:00+00:00,-0.018976,-0.043791,-0.005185,-0.034765,-0.030293,-0.006440,0.011090,1.449818,41.701895,0.005654,0.027629,0.019412,NaN,NaN
1997,2026-11-04 00:00:00+00:00,0.053058,0.016520,0.038138,0.014820,0.019246,0.043801,0.016206,1.934408,57.779156,0.006763,0.038654,0.023246,NaN,NaN


Labeled rows: 1995
Target distribution:
target
1.0    1160
0.0     835
NaN       5
Name: count, dtype: int64


## 5. Train and evaluate
Chronological holdout with a horizon purge. Models: dummy baseline and histogram gradient boosting. The test set is exploratory; do not repeatedly optimize against it and call it unbiased.

In [6]:
models, metadata = train_experiment(prices, spec, SYMBOL, INTERVAL)
display(pd.DataFrame(metadata["metrics"]).T)
print("Training rows:", metadata["train_rows"], "Test rows:", metadata["test_rows"])
print("Test window:", metadata["test_start"], "to", metadata["test_end"])


,accuracy,balanced_accuracy,brier_score,roc_auc
prior_baseline,0.579747,0.500000,0.243672,0.500000
hist_gradient_boosting,0.529114,0.496935,0.276020,0.483375


Training rows: 1546 Test rows: 395
Test window: 2025-04-07 00:00:00+00:00 to 2026-10-30 00:00:00+00:00


## 5A. Model results inside the notebook

Run this section after **Train and evaluate**. These tables and charts are calculated from the current training run, not from saved Streamlit artifacts. Classification uses ROC-AUC (higher is better); regression uses MAE (lower is better).


In [ ]:
# Results for the current run: baseline versus trained model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

current_metrics = pd.DataFrame.from_dict(metadata["metrics"], orient="index")
current_metrics.index.name = "model"
metric_name = "mae" if spec.target_type == "return" else "roc_auc"
ascending = spec.target_type == "return"
ranked = current_metrics.sort_values(metric_name, ascending=ascending, na_position="last")
display(Markdown("### Current experiment — model leaderboard"))
display(ranked.style.format(precision=4))
print(f"Stock: {SYMBOL} | Interval: {INTERVAL} | Target: {spec.target_type} | Horizon: {spec.horizon} bars")
print(f"Train: {metadata['train_rows']:,} rows | Test: {metadata['test_rows']:,} rows")
print(f"Holdout: {metadata['test_start']} through {metadata['test_end']}")
valid_scores = ranked[metric_name].dropna()
if len(valid_scores):
    print(f"Top observed model on this holdout ({metric_name}): {valid_scores.index[0]}")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    valid_scores.plot.bar(ax=ax)
    ax.set_ylabel(metric_name.upper())
    ax.set_title("Current experiment: holdout model comparison")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print(f"{metric_name} is not available for this evaluation window.")
print("Exploratory holdout only: selecting the best score on this set biases evaluation.")


## 5B. Target and feature diagnostics

Inspect the exact target distribution and missing-feature rates before trusting the score. This uses the experiment frame generated above.


In [ ]:
# Feature coverage and label distribution
feature_missing = frame[list(spec.features)].isna().mean().mul(100).sort_values(ascending=False)
display(Markdown("### Missing values by selected feature (%)"))
display(feature_missing.rename("missing_pct").to_frame().style.format({"missing_pct": "{:.2f}%"}))
display(Markdown("### Target distribution"))
if spec.target_type == "return":
    display(frame["target"].dropna().describe().to_frame("forward_return"))
    fig, ax = plt.subplots(figsize=(8, 3.5))
    frame["target"].dropna().hist(bins=40, ax=ax)
    ax.set_title("Forward return target distribution")
    ax.set_xlabel("Future fractional return")
    plt.tight_layout()
    plt.show()
else:
    counts = frame["target"].dropna().astype(int).value_counts().sort_index()
    display(pd.DataFrame({"count": counts, "fraction": counts / counts.sum()}))
    fig, ax = plt.subplots(figsize=(6, 3))
    counts.plot.bar(ax=ax)
    ax.set_title("Classification target distribution")
    ax.set_xlabel("Class")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()


## Feature engineering extension guide

The notebook calls `build_experiment_frame()` in `src/stock_ml_dynamic/experiments.py`, controlled by `ExperimentSpec` and `AVAILABLE_FEATURES` in `experiment.py`. `features.py` is the original fixed-feature implementation and is **not** the configurable notebook engine.

To add an indicator, implement a causal calculation in `experiments.py`, register its name in `AVAILABLE_FEATURES`, add any tunable window to `ExperimentSpec`, and write a test ensuring future-price changes do not affect earlier features. Then select it in the notebook `FEATURES` list. For a new target, update target construction, validation, and training task type together. Do not include `target` or `target_forward_return` in model inputs.

The current notebook has one chronological holdout with a horizon purge. Before deployment, add walk-forward validation and a final untouched holdout.


## 6. Save this experiment
Set SAVE_RUN to True only when you want this run persisted. Model artifacts and metadata are saved under `artifacts/experiments/`.

In [7]:
SAVE_RUN = True
if SAVE_RUN:
    output = save_experiment(models, metadata)
    print("Saved:", output)
else:
    print("Not saved")


Saved: artifacts/experiments/20261109T200116_527cbed5


## 7. Compare saved experiments
Only compare ROC-AUC when the target and evaluation windows match. Repeated tuning on a holdout biases estimates.

In [8]:
records = list_experiments()
results = pd.DataFrame(comparison_rows(records, symbol=SYMBOL, interval=INTERVAL))
if not results.empty:
    metric = "mae" if TARGET_TYPE == "return" else "roc_auc"
    comparable = results[(results["target"] == TARGET_TYPE) & (results["horizon"] == HORIZON)].copy()
    if metric in comparable:
        display(comparable.sort_values(metric, ascending=(metric == "mae"), na_position="last"))
    else:
        display(comparable)
else:
    print("No saved experiments yet")


,run_id,symbol,interval,target,horizon,spec_hash,model,accuracy,balanced_accuracy,brier_score,roc_auc
0,20261109T200116_527cbed5,AAPL,1d,direction,5,9f346e763ebf,prior_baseline,0.579747,0.500000,0.243672,0.500000
1,20261109T200116_527cbed5,AAPL,1d,direction,5,9f346e763ebf,hist_gradient_boosting,0.529114,0.496935,0.276020,0.483375


## 8. Iterate
Change FEATURES, indicator windows, TARGET_TYPE, HORIZON or THRESHOLD, then rerun from cell 3 onward. Each saved run retains its specification and metrics. For production selection, add walk-forward validation and an untouched final holdout. The dashboard is read-only and does not train models.
